In [ ]:
import pandas as pd
import plotly.express as px
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display
from io import StringIO


def visualize_dataset_summary(summary: dict, save_pdf: str | None = None) -> str:
    """
    Visualize a dataset summary in Jupyter, save a full PDF report, and return a formatted docstring.

    Parameters
    ----------
    summary : dict
        Dictionary from summarize_dataset() with keys such as
        'duration_stats', 'duration_histogram', 'signal_coverage',
        'dataset_classes', 'class_distribution', 'extra_class_distribution'.
    save_pdf : str or None
        If provided, saves a complete multi-page PDF report to this path.
    figsize : tuple
        Figure size for plots.

    Returns
    -------
    str
        A formatted Python docstring summarizing the dataset.
    """
    dur_stats = summary.get("duration_stats", {})
    dur_hist = summary.get("duration_histogram", {})
    signal_cov = summary.get("signal_coverage", pd.DataFrame())
    dataset_classes = summary.get("dataset_classes", None)
    class_dist = summary.get("class_distribution", None)
    extra_class_dist = summary.get("extra_class_distribution", None)

    # -------------------------------------------------------
    # 1️⃣ Construct docstring
    # -------------------------------------------------------
    buf = StringIO()
    buf.write('"""Dataset Summary\n')
    buf.write("Summary of core statistics for this dataset.\n\n")

    if dur_stats:
        buf.write("Durations:\n")
        for k, v in dur_stats.items():
            buf.write(f"    {k:<8}: {v}\n")

    if isinstance(signal_cov, pd.DataFrame) and not signal_cov.empty:
        buf.write("\nChannels (top 50):\n")
        for i, (ch, row) in enumerate(signal_cov.head(50).iterrows(), 1):
            buf.write(f"    {i:2d}. {ch:<20} {row['coverage[%]']:.1f}%\n")

    if dataset_classes:
        buf.write("\nClasses:\n")
        for c in dataset_classes:
            buf.write(f"    - {c}\n")

    if class_dist:
        buf.write("\nClass Frequencies:\n")
        for k, v in sorted(class_dist.items(), key=lambda x: x[1], reverse=True):
            buf.write(f"    {k:<15}: {v}\n")

    if extra_class_dist:
        buf.write("\nExtra Class Frequencies:\n")
        for k, v in sorted(extra_class_dist.items(), key=lambda x: x[1], reverse=True):
            buf.write(f"    {k:<15}: {v}\n")

    buf.write('"""\n')
    docstring = buf.getvalue()

    # -------------------------------------------------------
    # 2️⃣ Display inline in Jupyter
    # -------------------------------------------------------
    print("⏱ Duration Statistics")
    if dur_stats:
        display(pd.DataFrame([dur_stats]).T.rename(columns={0: "Value"}))
    else:
        print("No duration stats available.\n")

    if isinstance(signal_cov, pd.DataFrame) and not signal_cov.empty:
        print("\n📡 Channel Coverage (Top 50)")
        display(signal_cov.head(50))

    if dataset_classes:
        print("\n📚 Declared Dataset Classes:")
        display(pd.DataFrame(dataset_classes, columns=["Class"]))

    if class_dist:
        print("\n📊 Event Class Distribution")
        df = pd.DataFrame.from_dict(class_dist, orient="index", columns=["count"]).sort_values("count", ascending=False)
        display(df)

    if extra_class_dist:
        print("\n🔍 Extra Event Class Distribution")
        df_extra = pd.DataFrame.from_dict(extra_class_dist, orient="index", columns=["count"]).sort_values("count", ascending=False)
        display(df_extra)

    # -------------------------------------------------------
    # 3️⃣ PDF: include text, tables, plots
    # -------------------------------------------------------
    pdf = PdfPages(save_pdf) if save_pdf else None

    def show_and_save(fig):
        """Display the figure in Jupyter and save it to the PDF."""
        if pdf:
            pdf.savefig(fig, bbox_inches="tight")
        display(fig)
        plt.close(fig)

    # --- Cover page ---
    if pdf:
        fig, ax = plt.subplots(figsize=(8.3, 11.7))
        ax.axis("off")
        ax.text(0.05, 0.95, "Dataset Summary", fontsize=18, weight="bold", va="top")
        ax.text(0.05, 0.90, docstring, fontsize=9, family="monospace", va="top")
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

    # --- Duration histogram ---
    if dur_hist:
        fig, ax = plt.subplots(figsize=(10, 4))
        x, y = list(dur_hist.keys()), list(dur_hist.values())
        ax.bar(x, y, color="#4C72B0")
        ax.set_xticklabels(x, rotation=45, ha="right")
        ax.set_ylabel("Count")
        ax.set_title("Recording Duration Distribution")
        fig.tight_layout()
        show_and_save(fig)

    # --- Signal coverage ---
    if isinstance(signal_cov, pd.DataFrame) and not signal_cov.empty:
        top_cov = signal_cov.sort_values("coverage[%]", ascending=False).head(50)
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.barh(top_cov.index, top_cov["coverage[%]"], color="#55A868")
        ax.invert_yaxis()
        ax.set_xlabel("Coverage [% of files]")
        ax.set_title("Most Frequent Channels")
        fig.tight_layout()
        show_and_save(fig)

    # --- Class distributions ---
    if class_dist:
        df = pd.DataFrame.from_dict(class_dist, orient="index", columns=["count"]).sort_values("count", ascending=False)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(df.index, df["count"], color="#C44E52")
        ax.set_xticklabels(df.index, rotation=45, ha="right")
        ax.set_ylabel("Count")
        ax.set_title("Event Class Distribution")
        fig.tight_layout()
        show_and_save(fig)

    # --- Extra class distributions ---
    if extra_class_dist:
        df_extra = pd.DataFrame.from_dict(extra_class_dist, orient="index", columns=["count"]).sort_values("count", ascending=False)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(df_extra.index, df_extra["count"], color="#8172B3")
        ax.set_xticklabels(df_extra.index, rotation=45, ha="right")
        ax.set_ylabel("Count")
        ax.set_title("Extra Event Class Distribution")
        fig.tight_layout()
        show_and_save(fig)

    if pdf:
        pdf.close()
        print(f"📄 Saved full summary PDF to: {save_pdf}")

    return docstring


In [ ]:
from sleepwalker.datasets.CAP import CAP
from sleepwalker.datasets.MNC import MNC
from sleepwalker.datasets.NCHSDB import NCHSDB
from sleepwalker.datasets.SHHS import SHHS
from sleepwalker.datasets.ABC import ABC
from sleepwalker.datasets.ISRUC import ISRUC
from sleepwalker.datasets.Ruhrlandklinik import Ruhrlandklinik
from sleepwalker.datasets.SleepEDFx import SleepEDFx
from sleepwalker.datasets.SVUH_UCD import SVUH_UCD
from sleepwalker.datasets.Apples import Apples
from sleepwalker.datasets.HCHS import HCHS
from sleepwalker.datasets.MROS import MROS
from sleepwalker.datasets.Numom2b import Numom2b
from sleepwalker.datasets.WSC import WSC
from sleepwalker.datasets.Stages import Stages

from sleepwalker.datasets.utils import get_edf_files_in_repo, summarize_dataset
from IPython.display import Markdown, display

import logging
from sleepwalker.utils import logger
logger.set_level(logging.ERROR)

datasets = [
    ("/raid/data/sleep-edfx", "-PSG.edf", "EEG Fpz-Cz", SleepEDFx),
    ("/raid/data/cap", ".edf", "Fp2-F4", CAP),
    ("/raid/data/ruhrlandklinik/raw/", ".edf", "C4", Ruhrlandklinik),
    ("/raid/data/ISRUC", ".edf", "E1-M2", ISRUC),
    ("/cephfs_projects/sleepwalker/shhs", ".edf", "EMG", SHHS),
    ("/cephfs_projects/sleepwalker/mros", ".edf", "C4", MROS),
    ("/cephfs_projects/sleepwalker/SVUH-UCD", ".rec", "EMG", SVUH_UCD),
    ("/cephfs_projects/sleepwalker/abc", ".edf", "C4", ABC),
    ("/cephfs_projects/sleepwalker/apples/polysomnography", ".edf", "EMG", Apples),
    ("/cephfs_projects/sleepwalker/hchs", ".edf", "A-Snore", HCHS),
    ("/cephfs_projects/sleepwalker/mnc/cnc", ".edf", "E2", MNC),
    ("/cephfs_projects/sleepwalker/nchsdb/sleep_data", ".edf", "EEG F4-M1", NCHSDB),
    ("/cephfs_projects/sleepwalker/numom2b", ".edf", "ECG", Numom2b),
    ("/cephfs_projects/sleepwalker/stages", ".edf", "C4", Stages),
    ("/cephfs_projects/sleepwalker/wsc", ".edf", "ECG", WSC)
]

for edf_path, ending, channel_name, clazz in datasets:
    clazz_name = f"{clazz}".split(".")[-1].replace(">","").replace("'","")
    display(Markdown(f"## 💤 **{clazz_name}**"))
    display(Markdown("---"))

    edf_files = get_edf_files_in_repo(edf_path, ending=ending, recursive=True)
    summary = summarize_dataset(clazz, edf_files, channel_name=channel_name, estimate_class_frequencies=False)

    pdf_path = f"./{clazz_name}_summary.pdf"
    docstring = visualize_dataset_summary(summary, save_pdf=pdf_path)

    display(Markdown("**📘 Auto-generated Docstring:**"))
    print(docstring)
    display(Markdown(f"✅ Summary saved for **{clazz_name}** → `{pdf_path}`"))
    display(Markdown("---"))



In [ ]:
# from sleepwalker.core.signal import fix_edf_header
# from sleepwalker.datasets.utils import get_edf_files_in_repo
import pandas as pd
from sleepwalker.core.signal import read_edf_meta
from sleepwalker.datasets.utils import get_edf_files_in_repo


edf_files = get_edf_files_in_repo("/raid/sleepwalker/ruhrlandklinik/raw/train-test-2023", recursive=True)
meta = read_edf_meta(edf_files[0])
df = pd.DataFrame({"fs":meta["fs"]}, index=meta["signals"])
df[df.index.isin(["PosAngle", "X Axis", "Y Axis", "Z Axis", "Elevation", "Activity"])]
# print(f"Found {len(edf_files)} files")

# for e in edf_files:
#     fix_edf_header(e,dry=False)

In [1]:
import pandas as pd
import plotly.express as px
from IPython.display import display
from tqdm.auto import tqdm

from sleepwalker.datasets.Ruhrlandklinik import Ruhrlandklinik
from sleepwalker.datasets.utils import get_edf_files_in_repo


def collect_event_lengths(dataset, edf_files, labels, extra=False, max_patients=None, desc=None):
    rows = []
    iterator = edf_files if max_patients is None else edf_files[:max_patients]
    iterator = tqdm(iterator, desc=desc or f"Scanning events ({'extra' if extra else 'main'})")

    for edf_path in iterator:
        event_df = dataset.get_extra_event_df(edf_path, pd.Timestamp("1970-01-01")) if extra else dataset.get_event_df(edf_path, pd.Timestamp("1970-01-01"))
        if event_df is None or len(event_df) == 0:
            continue

        current = event_df[event_df["Label"].isin(labels)].copy()
        if len(current) == 0:
            continue

        if "Duration" in current.columns:
            duration_seconds = pd.to_numeric(current["Duration"], errors="coerce")
        else:
            duration_seconds = pd.Series(index=current.index, dtype=float)

        missing_duration = duration_seconds.isna()
        if missing_duration.any():
            duration_seconds.loc[missing_duration] = (
                current.loc[missing_duration, "Endtime"] - current.loc[missing_duration, "Starttime"]
            ).dt.total_seconds()

        current = current.assign(duration_seconds=duration_seconds).dropna(subset=["duration_seconds"])
        if len(current) == 0:
            continue

        rows.append(current[["Label", "duration_seconds"]])

    if len(rows) == 0:
        return pd.DataFrame(columns=["Label", "duration_seconds"])

    return pd.concat(rows, axis=0, ignore_index=True)


def summarize_event_lengths(all_events):
    if len(all_events) == 0:
        return pd.DataFrame(columns=["count", "mean_seconds", "min_seconds", "max_seconds", "mean", "min", "max"])

    summary = all_events.groupby("Label")["duration_seconds"].agg(["count", "mean", "min", "max"])
    summary = summary.rename(columns={
        "mean": "mean_seconds",
        "min": "min_seconds",
        "max": "max_seconds",
    })
    summary["mean"] = pd.to_timedelta(summary["mean_seconds"], unit="s")
    summary["min"] = pd.to_timedelta(summary["min_seconds"], unit="s")
    summary["max"] = pd.to_timedelta(summary["max_seconds"], unit="s")
    return summary.sort_values("count", ascending=False)


def plot_event_length_histogram(all_events, bins=50, title=None):
    if len(all_events) == 0:
        print("No matching events found.")
        return

    fig = px.histogram(
        all_events,
        x="duration_seconds",
        nbins=bins,
        color="Label" if all_events["Label"].nunique() > 1 else None,
        marginal="box",
        opacity=0.8,
        title=title or "Event length histogram",
        labels={"duration_seconds": "Event length [s]", "count": "Occurrences"},
    )
    fig.update_layout(bargap=0.05)
    fig.show()


ruhrland = Ruhrlandklinik(
    channels=[],
    sample_frequency=1,
    event_mapping={"LM": "lm"},
    remove_unmapped_events=False,
)
ruhrland_files = get_edf_files_in_repo("/raid/sleepwalker/ruhrlandklinik/raw/train-test-2023", recursive=True)
lm_events = collect_event_lengths(ruhrland, ruhrland_files, labels=["lm"], desc="Scanning Ruhrland LM events")
display(summarize_event_lengths(lm_events))
plot_event_length_histogram(lm_events, bins=60, title="Ruhrland LM event lengths")


/root/projects/sleepwalker/.venv/swa30/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Scanning Ruhrland LM events: 100%|██████████| 304/304 [00:22<00:00, 13.63it/s]


,count,mean_seconds,min_seconds,max_seconds,mean,min,max
Label,,,,,,,
lm,152726,1.96614,0.0,178.648,0 days 00:00:01.966140448,0 days,0 days 00:02:58.648000
